# VisClick — Part C (step 4): assemble source-domain training set

**Prerequisite:** unified Zenodo bundle is on Drive (`data/unified/{train,val,test}/{images,labels}/`). CLAY/VINS images optional (skipped automatically if absent).

**This notebook** (`VisClick_Detailed_Plan.md` §C.5):
1. Build a YOLO-format **source training set at `/content/source_train/`** (local, ephemeral). **No image data is duplicated on Drive** — images are exposed as **symlinks** pointing back to `data/unified/<split>/images/`.
2. **Remap labels** from the 12-class unified taxonomy → the 6 VisClick classes (`button, text, text_input, icon, menu, checkbox`). Only the tiny remapped `.txt` files are written.
3. Write `/content/source_train/data.yaml` (Ultralytics-ready, absolute path).
4. Patch the repo's `configs/yolo_source.yaml` at runtime so §D.1 training uses it directly.
5. Print per-split + per-class **`REPORT`** lines for `VisClick_Report_Data_Form.md` §2.1.

**Why local + symlinks?** Drive has limited free space; copying ~10k images would duplicate ~700 MB–4 GB. Symlinks under `/content/` avoid that.

**Session persistence.** After the first full build we save a **tiny bundle** (labels + manifest of image filenames, ~1–5 MB total) to `<DRIVE>/data/source_train_bundles/<split>.tar.gz`. On any *new* Colab session this cell **fast-restores** in seconds: extract labels from the bundle, read the manifest, and recreate symlinks into `unified/<split>/images/`. No 45-minute re-scan.

**Idempotent:** each split is skipped if `images/<split>/` already has the target number of files, or restored from the Drive bundle if present (or set `FORCE = True` to rebuild). If you already created a Drive copy from an earlier run, you can delete `<DRIVE>/data/source_train/` to reclaim space — this notebook no longer writes images there.

**Why random (not stratified)?** Reading 84k tiny label files on Drive FUSE is slow and timeout-prone. Random sampling on the unified train pool is a fine v1; the class distribution still follows the population. We log per-class counts so you can decide if a top-up from CLAY/VINS is needed later.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import os, random, shutil, json, time, subprocess
random.seed(0)

DRIVE   = "/content/drive/MyDrive/visclick"
DATA    = os.path.join(DRIVE, "data")
UNIFIED = os.path.join(DATA, "unified")

# Local working dir — symlinks point at Drive, so 0 image duplication on Drive.
OUT = "/content/source_train"
USE_SYMLINKS = True  # False = copy images (only useful if OUT is on Drive)

# Sample sizes — keeps Colab Free training tractable (see plan §C.5)
N = {"train": 8000, "val": 1000, "test": 1000}
FORCE = False  # set True to rebuild even if source_train/ is populated
MAX_IMG_BYTES = 6_000_000  # skip absurdly large images (val has multi-MB PNGs); 0 disables

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

# Unified bundle -> VisClick class mapping (per plan §C.3 table)
UNIFIED_NAMES = ["Button", "Text", "Image", "Icon", "Input", "Link",
                  "Checkbox", "Toggle", "Toolbar", "Navigation", "Modal", "Tab"]
UNIFIED_TO_VIS = {
    "Button": "button", "Toggle": "button", "Tab": "button",
    "Text": "text", "Link": "text",
    "Input": "text_input",
    "Image": "icon", "Icon": "icon",
    "Toolbar": "menu", "Navigation": "menu", "Modal": "menu",
    "Checkbox": "checkbox",
}
UNI_ID_TO_VIS_ID = {
    i: CLASSES.index(UNIFIED_TO_VIS[name]) for i, name in enumerate(UNIFIED_NAMES) if name in UNIFIED_TO_VIS
}
print("unified id -> visclick id =", UNI_ID_TO_VIS_ID)

for sub in ("images", "labels"):
    for sp in ("train", "val", "test"):
        os.makedirs(os.path.join(OUT, sub, sp), exist_ok=True)

print("OUT =", OUT)
print("FORCE =", FORCE, "| MAX_IMG_BYTES =", MAX_IMG_BYTES, "| target sizes =", N)

## 4.1 — Skip-all guard

If `source_train/images/<split>/` already has ≥0.9 × target files for **all three** splits, the assembly is skipped entirely.


In [ ]:
def _count(p):
    """Count regular files OR symlinks (treat broken symlinks as still counting)."""
    try:
        n = 0
        with os.scandir(p) as it:
            for e in it:
                if e.name.startswith("."):
                    continue
                if e.is_file(follow_symlinks=False) or e.is_symlink():
                    n += 1
        return n
    except OSError:
        return 0

have = {sp: _count(os.path.join(OUT, "images", sp)) for sp in N}
ok = all(have[sp] >= int(N[sp] * 0.9) for sp in N)
for sp in N:
    print(f"REPORT pre-check | split = {sp:5s} | have = {have[sp]:>5d} | target = {N[sp]:>5d}")

# --- fast-restore from Drive bundles (labels + manifest only; recreates symlinks) ---
BUNDLES = os.path.join(DATA, "source_train_bundles")
def _bundles_present():
    for sp in N:
        b = os.path.join(BUNDLES, f"{sp}.tar.gz")
        if not (os.path.isfile(b) and os.path.getsize(b) > 1000):
            return False
    return True

def _fast_restore():
    import tarfile
    for sp in N:
        t0r = time.time()
        bundle = os.path.join(BUNDLES, f"{sp}.tar.gz")
        with tarfile.open(bundle, "r:gz") as tf:
            tf.extractall(OUT)
        manifest = os.path.join(OUT, "manifests", f"{sp}.txt")
        if not os.path.isfile(manifest):
            print(f"  bundle for {sp} has no manifest; skipping restore")
            return False
        with open(manifest) as fh:
            names = [ln.strip() for ln in fh if ln.strip()]
        src_img_dir = os.path.join(UNIFIED, sp, "images")
        dst_img_dir = os.path.join(OUT, "images", sp)
        os.makedirs(dst_img_dir, exist_ok=True)
        made = 0
        for fn in names:
            dst = os.path.join(dst_img_dir, fn)
            if os.path.islink(dst) or os.path.isfile(dst):
                continue
            src = os.path.join(src_img_dir, fn)
            try:
                os.symlink(src, dst); made += 1
            except OSError:
                pass
        print(f"  restored {sp}: labels + {made} new symlinks (manifest {len(names)}) in {time.time()-t0r:0.1f}s")
    return True

if ok and not FORCE:
    print("REPORT step = ASSEMBLE | status = ALREADY_BUILT | path =", OUT)
    SKIP_ALL = True
elif _bundles_present() and not FORCE:
    print("REPORT step = ASSEMBLE | status = RESTORING_FROM_BUNDLE | dir =", BUNDLES)
    try:
        _fast_restore()
        print("REPORT step = ASSEMBLE | status = RESTORED_FROM_BUNDLE | path =", OUT)
        SKIP_ALL = True
    except Exception as e:
        print("fast-restore failed:", e, "-> falling through to full build")
        SKIP_ALL = False
else:
    SKIP_ALL = False
    print("REPORT step = ASSEMBLE | status = WILL_BUILD")

## 4.2 — Build each split (random sample + remap)

For each split this cell:
1. Lists the unified `labels/<split>/` directory once (small text files).
2. Shuffles + takes the first `N[split]` entries.
3. For each entry: finds the matching image, **rewrites the label** with mapped class IDs, then copies the image. Images larger than `MAX_IMG_BYTES` are skipped (val contains multi-MB PNGs) and replaced from the next sample.
4. Tracks per-class instance counts.

Skips a split if it already has ≥0.9 × target files (unless `FORCE=True`).


In [ ]:
def _list_labels_robust(d, cache_path, retries=4, sleep_s=5, find_timeout=240):
    """List *.txt names in `d`. Resilient to Drive FUSE timeouts via shell-find fallback + on-disk cache."""
    if os.path.isfile(cache_path):
        try:
            with open(cache_path) as fh:
                names = [ln.strip() for ln in fh if ln.strip()]
            if names:
                print(f"  cache hit: {cache_path} ({len(names)} entries)")
                return names
        except OSError:
            pass
    last_err = None
    for i in range(retries):
        try:
            names = [f for f in os.listdir(d) if f.endswith(".txt")]
            if names:
                with open(cache_path, "w") as fh:
                    fh.write("\n".join(names) + "\n")
                return names
        except OSError as e:
            last_err = e
            print(f"  listdir attempt {i+1}/{retries} failed: {e}; retrying in {sleep_s}s")
            time.sleep(sleep_s)
    try:
        print(f"  falling back to shell find on {d}")
        cmd = f"find '{d}' -maxdepth 1 -type f -name '*.txt' -printf '%f\\n' 2>/dev/null"
        r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=find_timeout)
        names = [ln.strip() for ln in r.stdout.splitlines() if ln.strip()]
        if names:
            with open(cache_path, "w") as fh:
                fh.write("\n".join(names) + "\n")
            print(f"  shell-find ok: {len(names)} entries")
            return names
    except Exception as e:
        last_err = e
    raise last_err if last_err else OSError("could not list " + d)

def _img_for_stem(img_dir, stem):
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(img_dir, stem + ext)
        if os.path.isfile(p):
            return p
    return None

def _remap_label_text(text):
    out_lines, n_kept = [], 0
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cid = int(parts[0])
        except ValueError:
            continue
        new_id = UNI_ID_TO_VIS_ID.get(cid)
        if new_id is None:
            continue
        out_lines.append(" ".join([str(new_id)] + parts[1:]))
        n_kept += 1
    return ("\n".join(out_lines) + ("\n" if out_lines else "")), n_kept

summary = {}
if SKIP_ALL:
    print("skipping all splits (already built).")
else:
    for split, target in N.items():
        dst_img = os.path.join(OUT, "images", split)
        dst_lbl = os.path.join(OUT, "labels", split)
        if not FORCE and _count(dst_img) >= int(target * 0.9):
            print(f"skip {split}: already has {_count(dst_img)} images")
            print(f"REPORT step = ASSEMBLE | split = {split} | status = SKIP_BUILT")
            continue

        src_img = os.path.join(UNIFIED, split, "images")
        src_lbl = os.path.join(UNIFIED, split, "labels")
        if not (os.path.isdir(src_img) and os.path.isdir(src_lbl)):
            print(f"REPORT step = ASSEMBLE | split = {split} | status = MISSING_UNIFIED")
            continue

        t0 = time.time()
        cache_path = f"/content/_unified_{split}_labels.list"
        all_labels = _list_labels_robust(src_lbl, cache_path)
        random.shuffle(all_labels)
        print(f"{split}: {len(all_labels)} candidate labels in {time.time()-t0:0.1f}s (cache: {cache_path})")

        per_class = {c: 0 for c in CLASSES}
        kept = 0
        skipped_big = skipped_empty = skipped_noimg = skipped_err = 0
        i = 0
        # take more than `target` because some are skipped
        while kept < target and i < len(all_labels):
            lname = all_labels[i]; i += 1
            stem = os.path.splitext(lname)[0]
            img_path = _img_for_stem(src_img, stem)
            if img_path is None:
                skipped_noimg += 1
                continue
            try:
                if MAX_IMG_BYTES and os.path.getsize(img_path) > MAX_IMG_BYTES:
                    skipped_big += 1
                    continue
                with open(os.path.join(src_lbl, lname), "r") as fh:
                    raw = fh.read()
            except OSError as e:
                skipped_err += 1
                continue
            new_text, n_kept_lines = _remap_label_text(raw)
            if n_kept_lines == 0:
                skipped_empty += 1
                continue
            try:
                dst_img_path = os.path.join(dst_img, os.path.basename(img_path))
                if not (os.path.isfile(dst_img_path) or os.path.islink(dst_img_path)):
                    if USE_SYMLINKS:
                        os.symlink(img_path, dst_img_path)
                    else:
                        shutil.copy2(img_path, dst_img_path)
                with open(os.path.join(dst_lbl, stem + ".txt"), "w") as fh:
                    fh.write(new_text)
            except OSError as e:
                skipped_err += 1
                continue
            for line in new_text.splitlines():
                cid = int(line.split()[0])
                per_class[CLASSES[cid]] += 1
            kept += 1
            if kept % 500 == 0:
                print(f"  {split}: {kept}/{target} kept ({time.time()-t0:0.0f}s)")

        elapsed = time.time() - t0
        summary[split] = {"kept": kept, "per_class": per_class,
                           "skipped_big": skipped_big, "skipped_empty": skipped_empty,
                           "skipped_noimg": skipped_noimg, "skipped_err": skipped_err,
                           "elapsed_s": round(elapsed, 1)}
        print(f"REPORT step = ASSEMBLE | split = {split} | kept = {kept} | target = {target} | elapsed_s = {elapsed:0.1f}")
        for c in CLASSES:
            print(f"REPORT step = ASSEMBLE | split = {split} | class = {c:11s} | n = {per_class[c]}")
        print(f"REPORT step = ASSEMBLE | split = {split} | skipped_big = {skipped_big} | skipped_empty = {skipped_empty} | skipped_noimg = {skipped_noimg} | skipped_err = {skipped_err}")

# --- Persist labels + manifest to Drive so future sessions fast-restore in seconds ---
import tarfile
if summary:
    os.makedirs(BUNDLES, exist_ok=True)
    mdir = os.path.join(OUT, "manifests")
    os.makedirs(mdir, exist_ok=True)
    for split in summary:
        img_dir = os.path.join(OUT, "images", split)
        names = sorted([f for f in os.listdir(img_dir) if not f.startswith(".")])
        mpath = os.path.join(mdir, f"{split}.txt")
        with open(mpath, "w") as fh:
            fh.write("\n".join(names) + "\n")
        tar_path = os.path.join(BUNDLES, f"{split}.tar.gz")
        with tarfile.open(tar_path, "w:gz") as tf:
            tf.add(os.path.join(OUT, "labels", split), arcname=f"labels/{split}")
            tf.add(mpath, arcname=f"manifests/{split}.txt")
        print(f"REPORT step = PERSIST | split = {split} | manifest = {len(names)} | bundle = {tar_path} | bytes = {os.path.getsize(tar_path)}")
    print("REPORT step = PERSIST | status = DONE | dir =", BUNDLES)

## 4.3 — Write `data.yaml` and patch repo config

Writes:
- `/content/source_train/data.yaml` (absolute path to the local dir; Ultralytics will read this in step 5).
- `/content/visclick/configs/yolo_source.yaml` patched in-place so the repo copy also points at the local path. The repo file in `git` keeps the `<DRIVE>` placeholder; this patch is **runtime-only** for the current session.


In [ ]:
import yaml as _yaml  # PyYAML is preinstalled in Colab; if not, !pip install pyyaml
data_yaml_path = os.path.join(OUT, "data.yaml")
data_yaml = {
    "path": OUT,
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(CLASSES),
    "names": CLASSES,
}
with open(data_yaml_path, "w") as fh:
    _yaml.safe_dump(data_yaml, fh, sort_keys=False)
print("REPORT step = ASSEMBLE | wrote =", data_yaml_path)
print(open(data_yaml_path).read())

repo_yaml = "/content/visclick/configs/yolo_source.yaml"
if os.path.isfile(repo_yaml):
    with open(repo_yaml, "w") as fh:
        _yaml.safe_dump(data_yaml, fh, sort_keys=False)
    print("REPORT step = ASSEMBLE | patched =", repo_yaml)
else:
    print("NOTE: repo yaml not found (was the repo cloned?):", repo_yaml)

## 4.4 — Final REPORT

Counts in the assembled set + paths. Copy these `REPORT` lines into `VisClick_Report_Data_Form.md` §2.1.


In [ ]:
totals = {c: 0 for c in CLASSES}
for split in ("train", "val", "test"):
    img_n = _count(os.path.join(OUT, "images", split))
    lbl_n = _count(os.path.join(OUT, "labels", split))
    print(f"REPORT final | split = {split:5s} | images = {img_n:>5d} | labels = {lbl_n:>5d}")
    if split in summary:
        for c in CLASSES:
            totals[c] += summary[split]["per_class"][c]
for c in CLASSES:
    print(f"REPORT final | class = {c:11s} | n_instances_total = {totals[c]}")
print("REPORT final | data.yaml =", os.path.join(OUT, "data.yaml"))
print("REPORT final | source_train =", OUT)

**Next:** `notebooks/05_train_source.ipynb` (§D.1) — train YOLOv8s on `source_train/data.yaml` and write weights to `<DRIVE>/weights/baseline_source/...`.

**Optional later top-up** (if a class looks starved in the REPORT counts above):
- Add **CLAY**-mapped images once RICO `combined/` is reachable.
- Add **VINS** images once the external archive is on Drive.
